In [5]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter, defaultdict
import math

nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)


# PART A: DATA PREPROCESSING

def load_data(filepath):
    """Load Twitter data from CSV file."""
    df = pd.read_csv(filepath, encoding='latin-1', header=None,
                     names=['target', 'ids', 'date', 'flag', 'user', 'text'],
                     on_bad_lines='skip', engine='python', nrows=50000)
    return df['text'].tolist()


def tokenize_sentences(sentences):
    """Tokenize sentences and convert to lowercase."""
    return [[w.lower() for w in word_tokenize(sent)] for sent in sentences]


def train_test_split(sentences, test_ratio=0.2):
    """Split into train and test sets."""
    split_idx = int(len(sentences) * (1 - test_ratio))
    return sentences[:split_idx], sentences[split_idx:]


def get_word_counts(train_data):
    """Count word frequencies in training data."""
    counts = Counter()
    for sent in train_data:
        counts.update(sent)
    return counts


def get_closed_vocabulary(word_counts, threshold=5):
    """Get vocabulary with words appearing >= threshold times."""
    return {word for word, count in word_counts.items() if count >= threshold}


def replace_oov(sentences, vocab, unk_token='<unk>'):
    """Replace out-of-vocabulary words with unknown token."""
    return [[w if w in vocab else unk_token for w in sent] for sent in sentences]


def preprocess(filepath, threshold=5, test_ratio=0.2):
    """Complete preprocessing pipeline."""
    print(f"Loading data from {filepath}...")
    data = load_data(filepath)
    print(f"Loaded {len(data)} tweets")

    print("Tokenizing sentences...")
    tokenized = tokenize_sentences(data)

    print("Splitting train/test sets...")
    train, test = train_test_split(tokenized, test_ratio)

    print("Building vocabulary...")
    word_counts = get_word_counts(train)
    vocab = get_closed_vocabulary(word_counts, threshold)

    print("Replacing OOV words...")
    train = replace_oov(train, vocab)
    test = replace_oov(test, vocab)

    return train, test, vocab


# PART B: N-GRAM LANGUAGE MODELS

def count_ngrams(sentences, n, start_token='<s>', end_token='<e>'):
    """Count n-grams from sentences."""
    counts = defaultdict(int)

    for sent in sentences:
        padded = [start_token] * (n - 1) + sent + [end_token]

        for i in range(len(padded) - n + 1):
            counts[tuple(padded[i:i+n])] += 1

    return dict(counts)


def estimate_probability(word, prev_ngram, ngram_counts, prev_ngram_counts, vocab_size, k=1.0):
    """Estimate probability with k-smoothing."""
    full_ngram = prev_ngram + (word,)
    numerator = ngram_counts.get(full_ngram, 0) + k
    denominator = prev_ngram_counts.get(prev_ngram, 0) + k * vocab_size
    return numerator / denominator if denominator > 0 else 0


def estimate_all_probabilities(prev_ngram, vocabulary, ngram_counts, prev_ngram_counts, vocab_size, k=1.0):
    """Estimate probabilities for all words."""
    probs = {}
    for word in vocabulary:
        probs[word] = estimate_probability(word, prev_ngram, ngram_counts, prev_ngram_counts, vocab_size, k)
    return probs


def make_count_matrix(ngram_counts, vocabulary, n):
    """Create count matrix for n-grams."""
    matrix = {}
    for ngram, count in ngram_counts.items():
        if len(ngram) == n:
            prev = ngram[:-1]
            if prev not in matrix:
                matrix[prev] = {}
            matrix[prev][ngram[-1]] = count
    return matrix


def make_probability_matrix(ngram_counts, prev_ngram_counts, vocabulary, k=1.0):
    """Create probability matrix for n-grams."""
    vocab_size = len(vocabulary)
    matrix = {}

    for prev_ngram in prev_ngram_counts.keys():
        matrix[prev_ngram] = estimate_all_probabilities(
            prev_ngram, vocabulary, ngram_counts, prev_ngram_counts, vocab_size, k
        )

    return matrix


# PART C: PERPLEXITY EVALUATION

def calculate_perplexity(sentences, ngram_counts, prev_ngram_counts, vocabulary, n, k=1.0):
    """Calculate perplexity score."""
    vocab_size = len(vocabulary)
    total_log_prob = 0.0
    total_words = 0

    for sent in sentences:
        padded = ['<s>'] * (n - 1) + sent + ['<e>']

        for i in range(n - 1, len(padded)):
            prev_ngram = tuple(padded[i-n+1:i])
            word = padded[i]

            full_ngram = prev_ngram + (word,)
            numerator = ngram_counts.get(full_ngram, 0) + k
            denominator = prev_ngram_counts.get(prev_ngram, 0) + k * vocab_size

            prob = numerator / denominator if denominator > 0 else k / (k * vocab_size)
            total_log_prob += math.log(prob)
            total_words += 1

    if total_words == 0:
        return float('inf')

    return math.exp(-total_log_prob / total_words)


# PART D: AUTO-COMPLETE SYSTEM

def get_suggestions(sentence, ngram_models, vocabulary, k=1.0, start_with=None, top_k=5):
    """Get word suggestions from n-gram models."""
    tokens = [w.lower() for w in word_tokenize(sentence)]
    all_suggestions = {}

    for n in sorted(ngram_models.keys()):
        ngram_counts, prev_ngram_counts = ngram_models[n]

        prev_ngram = tuple(tokens[-(n-1):]) if len(tokens) >= n-1 else tuple(['<s>'] * (n-1-len(tokens)) + tokens)

        for word in vocabulary:
            if start_with and not word.startswith(start_with):
                continue

            full_ngram = prev_ngram + (word,)
            numerator = ngram_counts.get(full_ngram, 0) + k
            denominator = prev_ngram_counts.get(prev_ngram, 0) + k * len(vocabulary)
            prob = numerator / denominator if denominator > 0 else k / (k * len(vocabulary))

            if word not in all_suggestions:
                all_suggestions[word] = prob
            else:
                all_suggestions[word] = max(all_suggestions[word], prob)

    ranked = sorted(all_suggestions.items(), key=lambda x: x[1], reverse=True)
    return [w for w, _ in ranked[:top_k]]


def main(data_path, threshold=5, test_ratio=0.2):
    """Execute complete pipeline."""

    print("NLP AUTO-COMPLETE SYSTEM")

    print("\n[PART A] Preprocessing Data...")
    train_data, test_data, vocabulary = preprocess(data_path, threshold, test_ratio)
    print(f"Train: {len(train_data)}, Test: {len(test_data)}, Vocab: {len(vocabulary)}")

    print("\n[PART B] Building N-gram Models...")
    ngram_models = {}

    for n in range(1, 7):
        ngram_counts = count_ngrams(train_data, n)
        prev_counts = defaultdict(int)
        for ngram in ngram_counts:
            prev_counts[ngram[:-1]] += ngram_counts[ngram]
        ngram_models[n] = (ngram_counts, dict(prev_counts))
        print(f"  {n}-gram: {len(ngram_counts)} unique n-grams")

    print("\n[PART C] Evaluating Models...")
    for n in range(1, 7):
        ngram_counts, prev_counts = ngram_models[n]
        perp = calculate_perplexity(test_data, ngram_counts, prev_counts, vocabulary, n)
        print(f"  {n}-gram perplexity: {perp:.4f}")

    print("\n[PART D] Auto-Complete System...")
    test_sentences = ["i love", "how are you", "the weather", "good morning", "i am"]

    for sent in test_sentences:
        sugg = get_suggestions(sent, ngram_models, vocabulary, top_k=5)
        print(f"  '{sent}' → {sugg}")


    print("Complete")

    return train_data, test_data, vocabulary, ngram_models


if __name__ == "__main__":
    train_data, test_data, vocabulary, ngram_models = main('training.1600000.processed.noemoticon.csv')

NLP AUTO-COMPLETE SYSTEM

[PART A] Preprocessing Data...
Loading data from training.1600000.processed.noemoticon.csv...
Loaded 50000 tweets
Tokenizing sentences...
Splitting train/test sets...
Building vocabulary...
Replacing OOV words...
Train: 40000, Test: 10000, Vocab: 5901

[PART B] Building N-gram Models...
  1-gram: 5903 unique n-grams
  2-gram: 155013 unique n-grams
  3-gram: 385896 unique n-grams
  4-gram: 519657 unique n-grams
  5-gram: 567305 unique n-grams
  6-gram: 580894 unique n-grams

[PART C] Evaluating Models...
  1-gram perplexity: 319.5865
  2-gram perplexity: 289.5531
  3-gram perplexity: 1256.3483
  4-gram perplexity: 2408.4584
  5-gram perplexity: 2898.0852
  6-gram perplexity: 3031.9630

[PART D] Auto-Complete System...
  'i love' → ['you', 'to', 'the', 'it', 'my']
  'how are you' → ["'re", 'have', '!', 'are', '.']
  'the weather' → ['is', '.', '!', 'in', ',']
  'good morning' → ['.', '!', ',', '...', 'and']
  'i am' → ['i', 'so', 'not', 'going', 'in']
Complete
